<a href="https://colab.research.google.com/github/guitorte/audio/blob/claude/stem-midi-converter-OwEoi/stem-to-midi/notebooks/Stem_to_MIDI_MVP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Stem → MIDI — MVP (single track)

Converte **um único stem isolado** (saída do Demucs) em um arquivo MIDI.

Dois motores, escolhidos automaticamente pelo `STEM_TYPE`:

| Stem | Engine | Notas |
|---|---|---|
| `vocals`, `bass`, `guitar`, `piano`, `other` | [Spotify Basic Pitch](https://github.com/spotify/basic-pitch) (Apache-2.0, ONNX) | pitched, polifônico |
| `drums` | [ADTOF-pytorch](https://github.com/xavriley/ADTOF-pytorch) (port de [ADTOF](https://github.com/MZehren/ADTOF), AGPL-3.0) | kick/snare/hat/tom/cymbals em GM drum map |

## Convenção de paths no Drive

| | Path |
|---|---|
| Entrada | `/content/drive/MyDrive/stem-to-midi/input/stem.wav` |
| Saída   | `/content/drive/MyDrive/stem-to-midi/output/stem.mid` |


## 1. Instalar dependências

⚠️ Após rodar esta célula, **reinicie o runtime** (`Runtime ▸ Restart Session`) e siga das células seguintes.

In [ ]:
# IMPORTANTE: Colab roda Python 3.12. basic-pitch 0.4.0 puxa tensorflow
# <2.15.1 / tflite-runtime que não têm wheel para 3.12. Workaround:
# instalar basic-pitch --no-deps e cair no backend ONNX (modelo nmp.onnx
# vem embutido). Ver spotify/basic-pitch#188.
#
# ADTOF-pytorch é PyTorch-only e tem pesos embutidos — instala limpo.

!pip uninstall -y basic-pitch tensorflow tflite-runtime 2>/dev/null
!pip install basic-pitch --no-deps onnxruntime
!pip install "resampy<0.4.3" librosa pretty_midi mir_eval scikit-learn scipy typing_extensions soundfile mido matplotlib flatbuffers protobuf
!pip install git+https://github.com/xavriley/ADTOF-pytorch.git

# Verificação
import importlib.util
_required = ('basic_pitch', 'onnxruntime', 'adtof_pytorch', 'pretty_midi', 'librosa', 'soundfile', 'mido')
_missing = [m for m in _required if importlib.util.find_spec(m) is None]
print('=' * 60)
if _missing:
    print(f'Módulos não encontrados após install: {_missing}')
    print('   A célula 2 tentará reinstalar automaticamente.')
else:
    print('Instalação concluída — Basic Pitch (ONNX) + ADTOF-pytorch OK.')
print('Reinicie o runtime (Runtime > Restart Session) e siga adiante.')
print('=' * 60)

## 2. Imports

In [ ]:
import os, sys, subprocess, importlib, warnings
import numpy as np
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

# Auto-instala módulos faltantes.
def _pip(*args):
    print('pip', *args)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *args])

def _ensure(module_name, *, pip_args=None):
    try:
        return importlib.import_module(module_name)
    except ImportError:
        pip_args = pip_args or [module_name.replace('_', '-')]
        _pip(*pip_args)
        importlib.invalidate_caches()
        return importlib.import_module(module_name)

# Backend pitched: ONNX (não dependemos de TF/TFLite — quebram em Py 3.12)
onnxruntime = _ensure('onnxruntime')

# basic-pitch precisa ser instalado com --no-deps para evitar puxar TF
try:
    _bp_inference = importlib.import_module('basic_pitch.inference')
except ImportError:
    _pip('basic-pitch', '--no-deps')
    _pip('resampy<0.4.3', 'librosa', 'pretty_midi', 'mir_eval',
         'scikit-learn', 'scipy', 'typing_extensions', 'flatbuffers', 'protobuf')
    importlib.invalidate_caches()
    _bp_inference = importlib.import_module('basic_pitch.inference')
bp_predict = _bp_inference.predict

# ADTOF-pytorch (drums)
try:
    from adtof_pytorch import transcribe_to_midi as adtof_transcribe
except ImportError:
    _pip('git+https://github.com/xavriley/ADTOF-pytorch.git')
    importlib.invalidate_caches()
    from adtof_pytorch import transcribe_to_midi as adtof_transcribe

librosa     = _ensure('librosa')
_ensure('librosa.display')
pretty_midi = _ensure('pretty_midi', pip_args=['pretty_midi'])
soundfile   = _ensure('soundfile')
mido        = _ensure('mido')

from IPython.display import Audio, display, FileLink

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f'Colab           : {IN_COLAB}')
print(f'ONNX runtime    : {onnxruntime.__version__}')
print(f'Engines prontas : Basic Pitch (pitched) + ADTOF-pytorch (drums)')

## 3. Montar Google Drive

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    print('Drive montado.')
else:
    print('Fora do Colab — pulando mount.')

## 4. Configurar paths e tipo de stem

Coloque o arquivo do stem em `/content/drive/MyDrive/stem-to-midi/input/stem.wav` (ou .mp3, .flac, .m4a) e selecione o tipo abaixo. O tipo carrega um preset de parâmetros do Basic Pitch ajustado para aquele instrumento (faixa de frequência, comprimento mínimo de nota, etc).

In [ ]:
# @title Parâmetros
BASE_DIR    = '/content/drive/MyDrive/stem-to-midi'  # @param {type:'string'}
INPUT_NAME  = 'stem.wav'                              # @param {type:'string'}
STEM_TYPE   = 'bass'  # @param ['vocals', 'bass', 'guitar', 'piano', 'drums', 'other']

INPUT_DIR  = os.path.join(BASE_DIR, 'input')
OUTPUT_DIR = os.path.join(BASE_DIR, 'output')
os.makedirs(INPUT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

AUDIO_PATH = os.path.join(INPUT_DIR, INPUT_NAME)
if not os.path.exists(AUDIO_PATH):
    stem_base = os.path.splitext(INPUT_NAME)[0]
    for ext in ('.wav', '.mp3', '.flac', '.m4a'):
        candidate = os.path.join(INPUT_DIR, stem_base + ext)
        if os.path.exists(candidate):
            AUDIO_PATH = candidate
            print(f'Arquivo encontrado com extensão alternativa: {os.path.basename(AUDIO_PATH)}')
            break

MIDI_PATH = os.path.join(OUTPUT_DIR, os.path.splitext(os.path.basename(AUDIO_PATH))[0] + '.mid')

print(f'Input audio : {AUDIO_PATH}  (existe: {os.path.exists(AUDIO_PATH)})')
print(f'Output MIDI : {MIDI_PATH}')
print(f'Stem type   : {STEM_TYPE}')

if not os.path.exists(AUDIO_PATH):
    print(f'\nColoque um arquivo em {INPUT_DIR}/ antes de seguir.')
else:
    size_mb = os.path.getsize(AUDIO_PATH) / 1024**2
    print(f'\nTamanho: {size_mb:.2f} MB')
    display(Audio(AUDIO_PATH))

## 5. Carregar presets por tipo de stem

In [ ]:
STEM_PRESETS = {
    'bass':   {'engine': 'basic_pitch', 'onset_threshold': 0.5, 'frame_threshold': 0.3, 'minimum_note_length':  80.0, 'minimum_frequency':  30.0, 'maximum_frequency':  350.0},
    'vocals': {'engine': 'basic_pitch', 'onset_threshold': 0.6, 'frame_threshold': 0.3, 'minimum_note_length': 100.0, 'minimum_frequency':  80.0, 'maximum_frequency': 1100.0},
    'guitar': {'engine': 'basic_pitch', 'onset_threshold': 0.5, 'frame_threshold': 0.3, 'minimum_note_length':  58.0, 'minimum_frequency':  70.0, 'maximum_frequency': 1500.0},
    'piano':  {'engine': 'basic_pitch', 'onset_threshold': 0.5, 'frame_threshold': 0.3, 'minimum_note_length':  58.0, 'minimum_frequency':  27.5, 'maximum_frequency': 4200.0},
    'drums':  {'engine': 'adtof'},
    'other':  {'engine': 'basic_pitch', 'onset_threshold': 0.5, 'frame_threshold': 0.3, 'minimum_note_length':  58.0},
}

preset = dict(STEM_PRESETS[STEM_TYPE])
ENGINE = preset.pop('engine')

print(f'Stem type : {STEM_TYPE}')
print(f'Engine    : {ENGINE}')
for k, v in preset.items():
    print(f'  {k:22s} = {v}')

## 6. Transcrever stem → MIDI

In [ ]:
assert os.path.exists(AUDIO_PATH), f'Coloque o stem em {AUDIO_PATH} primeiro.'

import time
print(f'Transcrevendo {os.path.basename(AUDIO_PATH)} com engine={ENGINE} ...')
_t0 = time.time()

if ENGINE == 'adtof':
    # ADTOF-pytorch: drum transcription com pesos embutidos.
    adtof_transcribe(AUDIO_PATH, MIDI_PATH)
    midi_data = pretty_midi.PrettyMIDI(MIDI_PATH)
    note_events = None
else:
    _model_output, midi_data, note_events = bp_predict(AUDIO_PATH, **preset)
    midi_data.write(MIDI_PATH)

_dt = time.time() - _t0

n_notes  = sum(len(i.notes) for i in midi_data.instruments)
midi_dur = midi_data.get_end_time()

print(f'\nTempo de inferência : {_dt:.1f}s')
print(f'Notas detectadas    : {n_notes}')
print(f'Duração do MIDI     : {midi_dur:.1f} s')
print(f'MIDI salvo em       : {MIDI_PATH}')

if n_notes > 0:
    all_notes = [n for inst in midi_data.instruments for n in inst.notes]
    pitches   = [n.pitch for n in all_notes]
    durations = [n.end - n.start for n in all_notes]
    velocities = [n.velocity for n in all_notes]
    print(f'\nPitch range      : {min(pitches)} – {max(pitches)} (MIDI)')
    print(f'Duração média    : {np.mean(durations):.3f} s')
    print(f'Velocity range   : {min(velocities)} – {max(velocities)}')

    if ENGINE == 'adtof':
        from collections import Counter
        GM_DRUMS = {35:'Acoustic Bass Drum', 36:'Kick', 38:'Snare', 39:'Hand Clap',
                    40:'Electric Snare', 41:'Low Floor Tom', 42:'Closed Hi-Hat',
                    43:'High Floor Tom', 44:'Pedal Hi-Hat', 45:'Low Tom',
                    46:'Open Hi-Hat', 47:'Low-Mid Tom', 48:'Hi-Mid Tom',
                    49:'Crash 1', 50:'High Tom', 51:'Ride 1'}
        print('\nDistribuição por elemento de bateria:')
        for pitch, count in sorted(Counter(pitches).items(), key=lambda x: -x[1]):
            name = GM_DRUMS.get(pitch, f'pitch {pitch}')
            print(f'  {name:22s} (MIDI {pitch:3d}): {count:4d} hits')

## 7. Piano roll do MIDI gerado

In [ ]:
if n_notes == 0:
    print('Nenhuma nota — pule esta célula.')
else:
    roll = midi_data.get_piano_roll(fs=10)
    pmin, pmax = max(0, min(pitches) - 2), min(127, max(pitches) + 2)
    roll_view = roll[pmin:pmax + 1, :]

    NOTE = ['C','C#','D','D#','E','F','F#','G','G#','A','A#','B']
    GM_DRUMS_SHORT = {35:'BD', 36:'Kick', 38:'Snare', 39:'Clap', 40:'ElSnr',
                      41:'LoFlT', 42:'HH', 43:'HiFlT', 44:'PdHH', 45:'LoT',
                      46:'OpHH', 47:'LMT', 48:'HMT', 49:'Crash', 50:'HiT', 51:'Ride'}

    fig, ax = plt.subplots(figsize=(14, 5))
    cmap = 'plasma' if ENGINE == 'adtof' else 'Blues'
    ax.imshow(roll_view, aspect='auto', origin='lower', cmap=cmap,
              interpolation='nearest',
              extent=[0, roll_view.shape[1] / 10, pmin, pmax])
    ax.set_xlabel('Tempo (s)')
    ax.set_ylabel('Drum element' if ENGINE == 'adtof' else 'Pitch MIDI')
    ax.set_title(f'Piano roll — {os.path.basename(MIDI_PATH)} ({STEM_TYPE}, {ENGINE})', fontweight='bold')

    if ENGINE == 'adtof':
        # Rotular cada pitch presente com nome GM
        unique_pitches = sorted(set(pitches))
        ax.set_yticks(unique_pitches)
        ax.set_yticklabels([GM_DRUMS_SHORT.get(p, f'p{p}') for p in unique_pitches])
    else:
        yt = list(range(pmin, pmax + 1, max(1, (pmax - pmin) // 12)))
        ax.set_yticks(yt)
        ax.set_yticklabels([f'{NOTE[p%12]}{p//12 - 1}' for p in yt])
    plt.tight_layout()
    plt.show()

## 8. Preview sonoro do MIDI (sintetizado)

In [ ]:
if n_notes == 0:
    print('Nenhuma nota — sem preview.')
else:
    sr_preview = 22050
    if ENGINE == 'adtof':
        # midi_data.synthesize() ignora canal 10 GM drums (não tem pitch tonal).
        # Sintetizar drums com noise burst rápido por hit.
        dur_total = midi_data.get_end_time()
        wave = np.zeros(int(dur_total * sr_preview) + sr_preview)
        all_notes = [n for inst in midi_data.instruments for n in inst.notes]
        for nt in all_notes:
            idx = int(nt.start * sr_preview)
            length = int(0.08 * sr_preview)
            env = np.exp(-np.linspace(0, 6, length))
            if nt.pitch in (35, 36):       # kick: baixa frequência
                tone = np.sin(2*np.pi*60*np.arange(length)/sr_preview)
            elif nt.pitch in (38, 40):     # snare: noise + tone
                tone = np.random.randn(length) * 0.5 + 0.3*np.sin(2*np.pi*200*np.arange(length)/sr_preview)
            else:                          # hats/cymbals/toms: noise high-pass-ish
                tone = np.random.randn(length) * 0.8
            wave[idx:idx+length] += tone * env * (nt.velocity/127.0)
        if np.max(np.abs(wave)) > 0:
            wave = 0.9 * wave / np.max(np.abs(wave))
    else:
        wave = midi_data.synthesize(fs=sr_preview)
        if np.max(np.abs(wave)) > 0:
            wave = 0.9 * wave / np.max(np.abs(wave))

    print('Áudio original (stem):')
    display(Audio(AUDIO_PATH))
    label = 'MIDI sintetizado (noise/tone bursts)' if ENGINE == 'adtof' else 'MIDI sintetizado (onda senoidal)'
    print(label + ':')
    display(Audio(wave, rate=sr_preview))

## 9. Download do MIDI

In [ ]:
print(f'Arquivo salvo em: {MIDI_PATH}')
display(FileLink(MIDI_PATH))

if IN_COLAB:
    try:
        from google.colab import files
        files.download(MIDI_PATH)
    except Exception as e:
        print(f'(download manual via Drive — botão automático falhou: {e})')